# 实验5_1: TensorFlow模型生成

## 1. 设置与导入库
首先导入 TensorFlow、Keras 以及其他辅助库。

In [1]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import PIL
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## 2. 下载并探索数据集

In [2]:
print("Downloading flower dataset...")
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_file_path = tf.keras.utils.get_file('flower_photos.tar', origin=dataset_url, extract=True)

# 兼容 Keras 2 和 Keras 3 的解压路径
data_dir = pathlib.Path(data_file_path).parent / 'flower_photos_extracted' / 'flower_photos'
if not data_dir.exists():
    data_dir = pathlib.Path(data_file_path).with_suffix('')

print("Dataset path resolved to:", data_dir)
image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Total number of images: {image_count}")

Dataset path resolved to: C:\Users\ljk18\.keras\datasets\flower_photos_extracted\flower_photos
Total number of images: 3670


## 3. 使用 Keras 效用函数加载数据
定义加载参数，并将 80% 的图像用于训练，将 20% 的图像用于验证。

In [3]:
batch_size = 32
img_height = 180
img_width = 180

print("Loading training dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

print("Loading validation dataset...")
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print("Class names:", class_names)

Loading training dataset...
Found 3670 files belonging to 5 classes.


Using 2936 files for training.


Loading validation dataset...


Found 3670 files belonging to 5 classes.


Using 734 files for validation.


Class names: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


## 4. 配置数据集以提高性能
使用缓冲预提取（prefetch）和缓存（cache）。

In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 5. 数据增强与随机失活

In [5]:
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(img_height,
                                  img_width,
                                  3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

D:\APP\Anaconda\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 6. 构建并编译模型

In [6]:
num_classes = len(class_names)

model = Sequential([
  data_augmentation,
  layers.Rescaling(1./255),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(num_classes, name="outputs")
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 180, 180, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 90, 90, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,965,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,989,285 (15.22 MB)

 Trainable params: 3,989,285 (15.22 MB)

 Non-trainable params: 0 (0.00 B)

## 7. 训练模型
对模型进行 15 个周期的训练。

In [ ]:
epochs = 15
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

## 8. 根据新数据进行预测验证
下载一张红向日葵图片，使用训练好的 Keras 模型进行预测，验证模型效果。

In [8]:
print("Downloading a test sunflower image for prediction...")
sunflower_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"
sunflower_path = tf.keras.utils.get_file('Red_sunflower', origin=sunflower_url)

img = tf.keras.utils.load_img(
    sunflower_path, target_size=(img_height, img_width)
)
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0) # Create a batch

predictions = model.predict(img_array)
score = tf.nn.softmax(predictions[0])

print(
    "Original model prediction: This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score)], 100 * np.max(score))
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


Original model prediction: This image most likely belongs to sunflowers with a 77.70 percent confidence.


## 9. 将 Keras 模型转换为 TensorFlow Lite 模型
使用 `TFLiteConverter` 将训练好的 Keras 模型转换为 TensorFlow Lite 格式，以便于在移动端和边缘设备上部署。

In [9]:
print("Converting Keras model to TensorFlow Lite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 保存模型到当前目录
tflite_model_path = 'model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)
print(f"TensorFlow Lite model saved successfully to {tflite_model_path}!")

Converting Keras model to TensorFlow Lite...


INFO:tensorflow:Assets written to: C:\Users\ljk18\AppData\Local\Temp\tmpe8xddd78\assets


INFO:tensorflow:Assets written to: C:\Users\ljk18\AppData\Local\Temp\tmpe8xddd78\assets


Saved artifact at 'C:\Users\ljk18\AppData\Local\Temp\tmpe8xddd78'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 180, 180, 3), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2202202846224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202202847952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202202847184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202202848336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202202848144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202202849104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202205045008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202205045584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202205046544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2202205045776: TensorSpec(shape=(), dtype=tf.resource, name=None)


TensorFlow Lite model saved successfully to model.tflite!


## 10. 运行 TensorFlow Lite 模型进行预测验证

In [ ]:
print("Verifying TensorFlow Lite model...")
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.get_signature_list()

classify_lite = interpreter.get_signature_runner('serving_default')
# 获取输入签名参数名
input_name = list(interpreter.get_signature_list()['serving_default']['inputs'])[0]
print(f"Lite model input name: {input_name}")

# 运行推理
predictions_lite_dict = classify_lite(**{input_name: img_array})
output_name = list(predictions_lite_dict.keys())[0]
predictions_lite = predictions_lite_dict[output_name]
score_lite = tf.nn.softmax(predictions_lite)

print(
    "Lite model prediction: This image most likely belongs to {} with a {:.2f} percent confidence."
    .format(class_names[np.argmax(score_lite)], 100 * np.max(score_lite))
)

print("Difference between original and lite predictions:", np.max(np.abs(predictions - predictions_lite)))
print("Model generation and verification completed successfully!")